# NYC Yellow Taxi — EDA Scratchpad

Early exploration: schema inspection, dirty-data audit, and sample-design
discovery. The polished analysis and management summary are in `analysis.ipynb`.

See `README.md` for environment setup and reproduction instructions.

## 1. Setup & Ingestion

Single in-memory source of truth: the CSV is read **once** with Polars, then
queried through **DuckDB** by referencing the frame directly (zero-copy). This
gives us a clean dataframe API *and* SQL over the same object, with no duplication.

In [1]:
import polars as pl
import duckdb

from pathlib import Path
_data = Path('data/Yellow_Taxi_Assignment.csv')
if not _data.exists():
    _data = Path('../data/Yellow_Taxi_Assignment.csv')
CSV_PATH = _data

# infer_schema_length raised well above the default 100: the fee columns are
# empty in early years and would be mis-typed from a short sample.
# try_parse_dates lets Polars type the pickup/dropoff columns as Datetime directly.
trips = pl.read_csv(CSV_PATH, infer_schema_length=100_000, try_parse_dates=True)

# duckdb resolves `trips` from local scope on every query below — no registration needed.
trips.shape

(304978, 19)

## 2. Schema & Types

First read of the contract: what each column *is*. Two early smells to confirm
later — `airport_fee` arrives as text (it's empty for most years, so no numeric
type was inferred), and `passenger_count` / `RatecodeID` are floats only because
nulls forced them off integer.

In [2]:
duckdb.sql("describe trips").pl()

column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""VendorID""","""BIGINT""","""YES""",null,null,null
"""tpep_pickup_datetime""","""TIMESTAMP""","""YES""",null,null,null
"""tpep_dropoff_datetime""","""TIMESTAMP""","""YES""",null,null,null
"""passenger_count""","""DOUBLE""","""YES""",null,null,null
"""trip_distance""","""DOUBLE""","""YES""",null,null,null
…,…,…,…,…,…
"""tolls_amount""","""DOUBLE""","""YES""",null,null,null
"""improvement_surcharge""","""DOUBLE""","""YES""",null,null,null
"""total_amount""","""DOUBLE""","""YES""",null,null,null


## 3. General Numbers (headline KPIs — *raw, pre-cleaning*)

A first sense of scale. These are computed on the **raw** data and therefore
include dirty rows (negative fares, a 177k-mile trip, etc.), so treat them as
upper-bound sanity checks, not final figures. We'll recompute on the cleaned
base once the cleaning policy is set.

In [3]:
duckdb.sql("""
with headline as (
    select
        count(*)                              as total_trips,
        count(distinct vendorid)              as vendors,
        min(tpep_pickup_datetime)::date       as first_pickup,
        max(tpep_pickup_datetime)::date       as last_pickup,
        sum(total_amount)                     as gross_revenue,
        avg(total_amount)                     as avg_total,
        avg(fare_amount)                      as avg_fare,
        avg(trip_distance)                    as avg_distance,
        avg(tip_amount)                       as avg_tip,
        avg(passenger_count)                  as avg_passengers
    from trips
),
final as (
    select
        total_trips,
        vendors,
        first_pickup,
        last_pickup,
        round(gross_revenue, 0) as gross_revenue,
        round(avg_total, 2)     as avg_total,
        round(avg_fare, 2)      as avg_fare,
        round(avg_distance, 2)  as avg_distance_mi,
        round(avg_tip, 2)       as avg_tip,
        round(avg_passengers, 2) as avg_passengers
    from headline
)
select * from final
""").pl()

total_trips,vendors,first_pickup,last_pickup,gross_revenue,avg_total,avg_fare,avg_distance_mi,avg_tip,avg_passengers
i64,i64,date,date,f64,f64,f64,f64,f64,f64
304978,5,2018-01-01,2023-01-31,5.809984e6,19.05,13.51,4.59,2.22,1.48


## 4. Column Profile — ranges, averages, nulls in one pass

`summarize` gives min / max / avg / approx-unique / null% per column at once —
exactly the "ranges + averages + dirtiness" view we want. Read the extremes
column by column: that's where the data-quality work announces itself.

In [4]:
# summarize is a built-in profiling command; one scan covers ranges, avgs and null%.
duckdb.sql("summarize trips").pl()

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""VendorID""","""BIGINT""","""1""","""6""",5,"""1.6727862337611237""","""0.5140690607643119""","""1""","""2""","""2""",304978,0.00
"""tpep_pickup_datetime""","""TIMESTAMP""","""2018-01-01 00:25:49""","""2023-01-31 23:57:28""",354846,"""2020-07-16 11:54:50.074786""",null,"""2019-04-06 08:10:22.271741""","""2020-07-16 19:22:53.82639""","""2021-10-22 10:58:16.055528""",304978,0.00
"""tpep_dropoff_datetime""","""TIMESTAMP""","""2018-01-01 00:38:59""","""2023-02-01 00:09:12""",374028,"""2020-07-16 12:11:31.905308""",null,"""2019-04-06 07:32:50.038582""","""2020-07-16 20:02:56.518833""","""2021-10-22 10:39:55.141042""",304978,0.00
"""passenger_count""","""DOUBLE""","""0.0""","""6.0""",7,"""1.4791261232294857""","""1.1082550217733613""","""1.0""","""1.0""","""1.667162271691571""",304978,3.12
"""trip_distance""","""DOUBLE""","""-16.86""","""177247.4""",2547,"""4.587588940841633""","""434.22662392916175""","""1.0152092647026207""","""1.7311710490417633""","""3.219556549374363""",304978,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""tolls_amount""","""DOUBLE""","""-13.75""","""96.55""",309,"""0.3907153302861036""","""1.6995466690622616""","""0.0""","""0.0""","""0.0""",304978,0.00
"""improvement_surcharge""","""DOUBLE""","""-1.0""","""1.0""",5,"""0.3127248522843342""","""0.11040399347159864""","""0.3""","""0.3""","""0.3""",304978,0.00
"""total_amount""","""DOUBLE""","""-198.55""","""561.49""",6705,"""19.05050059348644""","""15.464014063464806""","""10.851154568031726""","""14.475298728679396""","""20.75308074922922""",304978,0.00


## 5. How Dirty Is It? — explicit validity flags

`summarize` shows *distributions*; this quantifies *rule violations* against the
TLC data dictionary so we can size each cleaning decision before making it.

In [5]:
duckdb.sql("""
with flagged as (
    select
        count(*)                                                             as total_rows,
        count(*) filter (where fare_amount < 0)                              as negative_fare,
        count(*) filter (where total_amount < 0)                             as negative_total,
        count(*) filter (where trip_distance <= 0)                           as nonpositive_distance,
        count(*) filter (where trip_distance > 1000)                         as absurd_distance,
        count(*) filter (where passenger_count = 0)                          as zero_passengers,
        count(*) filter (where ratecodeid = 99)                              as ratecode_unknown,
        count(*) filter (where payment_type = 0)                             as payment_type_zero,
        count(*) filter (where tpep_dropoff_datetime <= tpep_pickup_datetime) as bad_duration
    from trips
),
final as (
    select
        total_rows,
        negative_fare,
        negative_total,
        nonpositive_distance, round(100.0 * nonpositive_distance / total_rows, 2) as nonpositive_distance_pct,
        absurd_distance,
        zero_passengers,      round(100.0 * zero_passengers      / total_rows, 2) as zero_passengers_pct,
        ratecode_unknown,
        payment_type_zero,    round(100.0 * payment_type_zero    / total_rows, 2) as payment_type_zero_pct,
        bad_duration
    from flagged
)
select * from final
""").pl()

total_rows,negative_fare,negative_total,nonpositive_distance,nonpositive_distance_pct,absurd_distance,zero_passengers,zero_passengers_pct,ratecode_unknown,payment_type_zero,payment_type_zero_pct,bad_duration
i64,i64,i64,i64,f64,i64,i64,f64,i64,i64,f64,i64
304978,1162,1167,3891,1.28,4,5535,1.81,268,9513,3.12,400


## 6. This Is a Multi-Year Sample, Not a Month

The decisive structural fact: the file holds ~60k trips per year for 2018–2022
plus a partial January 2023. That makes **year-over-year** the natural axis and
puts the **2020 demand shock** at the center of the story.

In [6]:
duckdb.sql("""
with yearly as (
    select
        year(tpep_pickup_datetime) as pickup_year,
        count(*)                   as trips
    from trips
    group by pickup_year
),
final as (
    select
        pickup_year,
        trips,
        round(100.0 * trips / sum(trips) over (), 2) as pct_of_sample
    from yearly
    order by pickup_year
)
select * from final
""").pl()

pickup_year,trips,pct_of_sample
i64,i64,f64
2018,59995,19.67
2019,59998,19.67
2020,59998,19.67
2021,59998,19.67
2022,59988,19.67
2023,5001,1.64
